# PyTorch: tensors, autograd, and neural nets that run as you write them

_PyTorch_

**A Python library where every operation runs immediately (eager / define-by-run), so you build, debug, and train models with ordinary Python — which is why research lives here.**

PyTorch is three ideas wearing one coat. Strip it down and a deep-learning framework is just:

       
         
- Tensors &mdash; a fast, n-dimensional number array (think NumPy) that can also run on a
         GPU and remember the operations performed on it.
         
- Automatic differentiation (autograd) &mdash; the machinery that, given any computation you wrote,
         computes the gradient of the result with respect to every input. This is dl-backprop done for
         you, automatically, for arbitrary code.
         
- Neural-network building blocks (torch.nn) &mdash; ready-made layers, activations,
         and losses you snap together into a model, plus optimizers (torch.optim) that apply the
         gradient updates.
       

---

This notebook is a practice scaffold for the **PyTorch: tensors, autograd, and neural nets that run as you write them** lesson. We build the idea up one small step at a time — run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Check the version and pick a device

PyTorch runs the same code on a CPU or a GPU; you just have to tell it which one to use. `torch.cuda.is_available()` returns `True` when a GPU runtime is attached. We pick **one** device string here and reuse it everywhere — mixing CPU and GPU tensors in a single op is the most common beginner error. We also set a seed so the random weights below come out the same every run.

In [ ]:
# Colab note: torch is preinstalled — no pip install needed. Just run this cell.
import torch
import torch.nn as nn

# torch.cuda.is_available() -> True if a GPU is present and usable.
print("PyTorch version:", torch.__version__)        # e.g. 2.x.x
print("CUDA available :", torch.cuda.is_available()) # True on a GPU runtime

# Pick ONE device string and use it everywhere (avoids CPU/GPU mismatch).
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device   :", device)

if device == "cuda":
    print("GPU name       :", torch.cuda.get_device_name(0))

# Set a seed so results are reproducible (random init, shuffling, dropout all use RNG).
torch.manual_seed(0)

### Step 2 — Make tensors and run an op eagerly

A tensor is PyTorch's n-dimensional number array — like a NumPy array, but it can live on a GPU and remember the operations performed on it. Each line below runs **immediately** (eager / define-by-run): there is no graph to compile and no session to open, so you can print any intermediate value the moment you compute it.

In [ ]:
# Make a couple of tensors and add them.
a = torch.tensor([1.0, 2.0, 3.0])   # a 1-D tensor (like a NumPy array)
b = torch.ones(3)                   # tensor([1., 1., 1.])
c = a + b                           # runs NOW -> tensor([2., 3., 4.])

print("a + b =", c)
print("shape :", c.shape, "| dtype:", c.dtype)

# Tensors interoperate with NumPy, but a tensor can also carry an autograd
# history AND live on a specific device — two things a NumPy array does not.
print("as numpy:", c.numpy())

### Step 3 — Move a tensor to the chosen device

`.to(device)` places a tensor on the device we picked in Step 1. On a GPU runtime the tensor's `.device` becomes `cuda:0`; on a CPU-only runtime it stays `cpu`. Keeping everything on the same device is what lets the next step's model and input meet without error.

In [ ]:
# .to(device) places the tensor on the chosen device.
a = a.to(device)
print("a is now on:", a.device)      # cpu, or cuda:0 on a GPU runtime

### Step 4 — Run a one-layer model forward (a teaser)

`nn.Linear(3, 1)` is a ready-made layer that computes `y = W x + b`; we move it to the **same** device as the input. Calling `model(x)` runs the forward pass immediately and returns a single predicted number. That is the whole skeleton — tensors flow into a model — and the rest of the course adds a loss, an optimizer, and a training loop around it.

In [ ]:
# nn.Linear(3, 1) computes y = W x + b. Move the model to the SAME device.
model = nn.Linear(in_features=3, out_features=1).to(device)

x = torch.tensor([1.0, 2.0, 3.0], device=device)   # one input on the right device
y = model(x)                                       # forward pass — runs immediately
print("model(x) =", y.item())                      # a single predicted number

# Skeleton: tensors -> model -> (add a loss + optimizer + loop) -> train.

## Visualize the data & results

_What does 'training' actually look like? A tiny model's loss falling over epochs — the curve you watch in every PyTorch run. Here: full-batch gradient descent fitting a line y = 2x + 1, the same loop PyTorch automates with autograd + an optimizer._

### Step 1 — Make a tiny noisy dataset

Before automating anything, we hand-roll the loop PyTorch does for you, so the moving parts are visible. We generate 64 points along the line `y = 2x + 1` and add a little noise. Our goal is to **recover** the slope `2` and intercept `1` from the noisy data using plain NumPy — the same job `nn.Linear` plus an optimizer would do.

In [ ]:
import numpy as np

# ---- A tiny dataset for a training loop done by hand ----
np.random.seed(0)
N = 64
x = np.linspace(-1, 1, N)
y_true = 2.0 * x + 1.0 + 0.15 * np.random.randn(N)   # line y = 2x + 1 + noise

w, b = 0.0, 0.0       # parameters PyTorch would store in nn.Linear
lr = 0.3              # learning rate (optimizer step size)
epochs = 40

### Step 2 — Run gradient descent by hand

Each epoch does the four steps every PyTorch run does: a **forward** pass to predict, an **MSE loss** to score it, a **backward** pass for the gradients, and a **step** that nudges the parameters downhill. Here the gradients are derived by calculus by hand — in PyTorch, `loss.backward()` computes them automatically via autograd, and `optimizer.step()` applies the update.

In [ ]:
losses = []

for ep in range(epochs):
    yhat = w * x + b                 # FORWARD pass  -> model(x)
    err = yhat - y_true
    loss = np.mean(err ** 2)         # MSE LOSS      -> loss = criterion(yhat, y)
    losses.append(round(float(loss), 4))

    # BACKWARD: hand-derived gradients. In PyTorch, loss.backward() does this
    # automatically via autograd — no calculus by hand.
    gw = 2 * np.mean(err * x)
    gb = 2 * np.mean(err)

    # STEP: gradient-descent update -> optimizer.step()
    w -= lr * gw
    b -= lr * gb

### Step 3 — Read off the recovered line and loss curve

After 40 epochs the parameters should sit close to the true `w = 2`, `b = 1`, and the loss should have fallen sharply from its starting value. The `(epoch, loss)` pairs trace the learning curve — the same falling curve you watch in every real PyTorch training run.

In [ ]:
print("recovered w, b:", round(w, 3), round(b, 3))   # -> 1.889 1.002
print("loss[0], loss[-1]:", losses[0], losses[-1])    # -> 2.2537 0.0211

points = [[ep, l] for ep, l in enumerate(losses)]
print(points)   # the (epoch, loss) pairs that trace the learning curve

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Import torch, then print three things: torch.__version__, torch.cuda.is_available(), and a device string set to "cuda" if a GPU is present else "cpu". Use this device variable everywhere in the rest of the session.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check for a GPU with torch.cuda.is_available(). — _It returns a boolean telling you whether a CUDA GPU runtime is attached._
- Pick the device once: device = "cuda" if torch.cuda.is_available() else "cpu". — _One variable reused everywhere prevents CPU/GPU mismatch errors later._

**Answer:** import torch
print(torch.__version__)              # e.g. 2.5.1+cu121
print(torch.cuda.is_available())      # True on a GPU runtime, else False
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)                         # cuda  (on a GPU runtime) or cpu

</details>

**Problem 2.** Type this in Colab. Create a = torch.tensor([1.0, 2.0, 3.0]) and b = torch.ones(3), then compute c = a + b. Predict c and c.shape before running, then print both to verify.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Add the two 1-D tensors with a + b. — _The op runs immediately (eager execution), so the result exists the instant you call it._
- Print c and c.shape. — _Verifies the elementwise sum and that the shape is unchanged at (3,)._

**Answer:** a = torch.tensor([1.0, 2.0, 3.0])
b = torch.ones(3)
c = a + b
print(c)            # tensor([2., 3., 4.])
print(c.shape)      # torch.Size([3])

</details>

**Problem 3.** Type this in Colab. Demonstrate eager execution: build x = torch.arange(4.), compute y = x * x, then print("mid-graph:", y) on the very next line, then compute z = y.sum() and print z. Notice you can print the intermediate y without compiling anything.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Print the intermediate tensor y right after computing it. — _Eager / define-by-run means real numbers exist at every line, so a plain print sees them._
- Reduce with y.sum() and print it. — _Shows the chain of ops each ran as Python reached it, no session or graph step._

**Answer:** x = torch.arange(4.)        # tensor([0., 1., 2., 3.])
y = x * x
print("mid-graph:", y)      # mid-graph: tensor([0., 1., 4., 9.])
z = y.sum()
print(z)                    # tensor(14.)

</details>

**Problem 4.** Type this in Colab. Make a tensor on the chosen device and move it back to the CPU. Set device as before, create t = torch.ones(2, 2, device=device), print t.device, then t = t.to("cpu") and print t.device again.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Create with device=device, then read t.device. — _Confirms the tensor was placed on the GPU (or CPU) you selected._
- Move with t.to("cpu") and reassign. — _.to(...) returns a tensor on the new device; you must reassign to keep it._

**Answer:** device = "cuda" if torch.cuda.is_available() else "cpu"
t = torch.ones(2, 2, device=device)
print(t.device)        # cuda:0 on a GPU runtime, else cpu
t = t.to("cpu")
print(t.device)        # cpu

</details>

**Problem 5.** Type this in Colab. Reproduce the famous device-mismatch error, then fix it. Create w = torch.ones(3, device=device) and x = torch.ones(3) (note: no device, so CPU). Try w + x on a GPU runtime and read the error; then fix it by moving x to device and print the sum.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Add a GPU tensor to a CPU tensor and read the RuntimeError. — _An op between tensors on different devices is forbidden — this is the #1 beginner error._
- Move both to one device: x = x.to(device), then add. — _Once every operand shares a device, the op is legal._

**Answer:** w = torch.ones(3, device=device)
x = torch.ones(3)                 # defaults to CPU
# On a GPU runtime, w + x raises:
# RuntimeError: Expected all tensors to be on the same device,
#   but found at least two devices, cuda:0 and cpu!
x = x.to(device)                  # fix: move x to the same device
print(w + x)                      # tensor([2., 2., 2.], device='cuda:0') on GPU
# On a CPU-only runtime both are already cpu and it just works.

</details>

**Problem 6.** Type this in Colab. Show that random init is reproducible with a seed. Call torch.manual_seed(0) then print(torch.randn(3)). Re-seed with torch.manual_seed(0) and print again — confirm you get the identical numbers.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set the seed with torch.manual_seed(0) before sampling. — _Fixes the RNG so the same draws come out every run — essential for reproducible experiments._
- Re-seed and re-sample. — _Resetting the seed rewinds the RNG, so the second draw matches the first._

**Answer:** torch.manual_seed(0)
print(torch.randn(3))   # tensor([ 1.5410, -0.2934, -2.1788])
torch.manual_seed(0)
print(torch.randn(3))   # tensor([ 1.5410, -0.2934, -2.1788])  identical

</details>

**Problem 7.** Type this in Colab. Build the five-line model teaser. Create model = torch.nn.Linear(3, 1) on the device, make an input x = torch.tensor([1.0, 2.0, 3.0], device=device), run a forward pass y = model(x), and print y.shape and y.item(). Predict the output shape first.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Move the model to the same device with .to(device). — _The layer's weights and the input must be on one device for the forward pass._
- Call model(x) (not model.forward(x)). — _Calling the module runs the forward pass plus hooks; nn.Linear(3,1) maps 3 inputs to 1 output._

**Answer:** torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.nn.Linear(3, 1).to(device)
x = torch.tensor([1.0, 2.0, 3.0], device=device)
y = model(x)
print(y.shape)     # torch.Size([1])  -- one output
print(y.item())    # a single float, e.g. -1.2353 (depends on seeded init)

</details>

**Problem 8.** Type this in Colab. Convert a tensor to a NumPy array and back. Create c = torch.tensor([2.0, 3.0, 4.0]), print c.numpy() and its type, then wrap a NumPy array back into a tensor with torch.from_numpy(np.array([5.0, 6.0])) and print it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bridge to NumPy with .numpy(). — _CPU tensors share their buffer with NumPy, so interop is cheap and common at the data boundary._
- Wrap a NumPy array with torch.from_numpy(...). — _Brings external array data into PyTorch as a tensor._

**Answer:** import numpy as np
c = torch.tensor([2.0, 3.0, 4.0])
arr = c.numpy()
print(arr, type(arr))                 # [2. 3. 4.] 
t = torch.from_numpy(np.array([5.0, 6.0]))
print(t)                              # tensor([5., 6.], dtype=torch.float64)

</details>